
## Table of Contents
1. Ranking Functions (ROW_NUMBER, RANK, DENSE_RANK)
2. Offset Functions (LAG, LEAD)
3. Running Totals and Moving Averages via Frames
4. FIRST_VALUE and LAST_VALUE Extractions

In [ ]:

# Global Imports and Runtime Visual Configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

# Visual styling settings
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10


In [ ]:
## 1. Ranking Functions (`ROW_NUMBER`, `RANK`, `DENSE_RANK`)
conn = sqlite3.connect(':memory:')
df_emp = pd.DataFrame({
    'department': ['IT', 'IT', 'IT', 'HR', 'HR', 'HR'],
    'name': ['Alice', 'Bob', 'Charlie', 'Dave', 'Eve', 'Frank'],
    'salary': [90000, 85000, 85000, 60000, 75000, 55000]
})
df_emp.to_sql('employees', conn, index=False, if_exists='replace')

query_rank = '''
    SELECT name, department, salary,
           ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) as row_num,
           RANK() OVER (PARTITION BY department ORDER BY salary DESC) as rk,
           DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) as dense_rk
    FROM employees
'''
res_rank = pd.read_sql(query_rank, conn)
print("Ranking Window Function Results:\n", res_rank)

# Visualization
plt.figure(figsize=(7, 3))
sns.barplot(x='name', y='salary', hue='department', data=res_rank, palette='viridis', errorbar=None)
plt.title('Employee Salary Ranking by Department')
plt.ylabel('Salary ($)')
plt.xticks(rotation=15)
plt.show

In [ ]:

## 2. Offset Functions (`LAG`, `LEAD`)

df_stock = pd.DataFrame({
    'date': ['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-04'],
    'price': [100, 105, 102, 110]
})
df_stock.to_sql('stock', conn, index=False, if_exists='replace')

query_offset = '''
    SELECT date, price,
           LAG(price, 1) OVER (ORDER BY date) as prev_price,
           price - LAG(price, 1) OVER (ORDER BY date) as price_change
    FROM stock
'''
res_offset = pd.read_sql(query_offset, conn)

In [ ]:
## 3. Running Totals and Moving Averages via Frames
df_rev = pd.DataFrame({
    'day': [1, 2, 3, 4, 5],
    'revenue': [100, 150, 200, 120, 300]
})
df_rev.to_sql('revenue_log', conn, index=False, if_exists='replace')

query_frame = '''
    SELECT day, revenue,
           SUM(revenue) OVER (ORDER BY day ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as running_total,
           AVG(revenue) OVER (ORDER BY day ROWS BETWEEN 1 PRECEDING AND CURRENT ROW) as moving_avg
    FROM revenue_log
'''
res_frame = pd.read_sql(query_frame, conn)
print(
    "Window Frame Aggregations:\n",
    res_frame,
)

In [ ]:
## 4. FIRST_VALUE and LAST_VALUE Extractions
df_subs = pd.DataFrame({
    'customer_id': [1, 1, 1, 2, 2],
    'month': [1, 2, 3, 1, 2],
    'tier': ['Basic', 'Silver', 'Gold', 'Silver', 'Gold']
})
df_subs.to_sql('subscriptions', conn, index=False, if_exists='replace')

query_fv = '''
    SELECT customer_id, month, tier,
           FIRST_VALUE(tier) OVER (PARTITION BY customer_id ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) as initial_tier
    FROM subscriptions
'''
res_fv = pd.read_sql(query_fv, conn)
print("First Value Extraction Results:\n", res_fv)